# Semantic Memory: Practice Exercise

Build an agent with semantic memory capabilities that can remember and retrieve user preferences about books and reading habits.

**What you'll implement:**
- Set up a memory store with semantic search capabilities
- Create memory management tools for the agent
- Build an agent that uses these tools alongside a book recommendation tool

**Estimated time:** 10-15 minutes

In [1]:
!pip install langchain-core langchain-openai langgraph langmem

In [2]:
import os
from functools import lru_cache

@lru_cache(maxsize=1)
def configure_environment(required_keys=None):
    """
    Factory function to configure environment variables.
    Executes once and caches results.
    """
    if required_keys is None:
        required_keys = ("OPENAI_API_KEY", )

    IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ

    if IN_COLAB:
        from google.colab import userdata
        print("Configuring for Google Colab environment...")
        for key in required_keys:
            try:
                os.environ[key] = userdata.get(key)
            except Exception:
                print(f"Warning: Could not find {key} in Colab secrets.")
    else:
        from dotenv import load_dotenv
        print("Configuring for local environment...")
        load_dotenv()

    # Validation
    for key in required_keys:
        if not os.getenv(key):
            raise ValueError(f"Missing required environment variable: {key}")

    return True

In [3]:
configure_environment(("OPENAI_API_KEY",))

Configuring for Google Colab environment...


True

## Setup

Run this cell to load dependencies and configure the environment.

In [4]:

from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.tools import tool
from langmem import create_manage_memory_tool, create_search_memory_tool
from langgraph.store.memory import InMemoryStore



# Use gpt-4o for reliable multi-step tool use (e.g., memory deletion requires search then delete)
model = ChatOpenAI(model="gpt-4o", temperature=0.1)

print("Setup complete!")

Setup complete!


## Context

You are building a book recommendation assistant that remembers user preferences across sessions. The assistant should:

- Remember favorite genres, authors, and reading preferences when users share them
- Retrieve relevant memories when making recommendations or answering questions
- Use a book lookup tool to find book information

**Input:** User messages about their reading preferences and book requests

**Output:** An agent that can save preferences to memory and retrieve them later using semantic search

**Requirements:**
- The memory store must use embeddings for semantic search
- Memory tools must use the namespace `("reading_preferences",)`
- The agent must have access to all three tools: manage memory, search memory, and book lookup

## Step 1: Create the Memory Store

Initialize an `InMemoryStore` with semantic search enabled using OpenAI embeddings.

In [5]:
def create_memory_store() -> InMemoryStore:
    """
    Create an in-memory store with semantic search capabilities.

    The store should be configured with an embedding model to enable
    similarity-based memory retrieval. Use OpenAI's text-embedding-3-small model.

    Returns:
        InMemoryStore: A memory store configured with embedding-based indexing
    """
    # TODO: Create and return an InMemoryStore with embedding configuration
    store = InMemoryStore(index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small",
        }
                          )
    return store


# Create the store
memory_store = create_memory_store()
print("Memory store created!" if memory_store else "Memory store not implemented yet")

Memory store created!


## Step 2: Create Memory Tools

Create the two memory management tools that the agent will use to save and retrieve memories.

In [6]:
def create_memory_tools(namespace: tuple):
    """
    Create memory management tools for the agent.

    Creates two tools:
    1. A tool to manage (create/update/delete) memories
    2. A tool to search for relevant memories

    Args:
        namespace: A tuple defining the memory namespace, e.g., ("reading_preferences",)

    Returns:
        tuple: (manage_tool, search_tool) - the two memory tools
    """
    # TODO: Create the manage memory tool using create_manage_memory_tool

    # TODO: Create the search memory tool using create_search_memory_tool

    manage_tool = create_manage_memory_tool(namespace=namespace)
    search_tool = create_search_memory_tool(namespace=namespace)
    return manage_tool, search_tool

# Create the memory tools with the specified namespace
memory_namespace = ("reading_preferences",)
tools_result = create_memory_tools(memory_namespace)

if tools_result:
    manage_tool, search_tool = tools_result
    print(f"Created tools: {manage_tool.name}, {search_tool.name}")
else:
    print("Memory tools not implemented yet")
    manage_tool, search_tool = None, None

Created tools: manage_memory, search_memory


## Step 3: Book Lookup Tool (Provided)

This tool is provided for you. The agent will use it alongside the memory tools.

In [7]:
# Book lookup tool - provided for you
@tool
def lookup_book(title: str) -> str:
    """Look up information about a book by title.

    Args:
        title: The title of the book to look up

    Returns:
        Information about the book including author, genre, and description
    """
    books = {
        "dune": "Dune by Frank Herbert - Science Fiction. An epic tale of politics, religion, and ecology on a desert planet.",
        "pride and prejudice": "Pride and Prejudice by Jane Austen - Romance/Classic. A witty exploration of love and social class in Regency England.",
        "the hobbit": "The Hobbit by J.R.R. Tolkien - Fantasy. A charming adventure of Bilbo Baggins and his unexpected journey.",
        "1984": "1984 by George Orwell - Dystopian Fiction. A chilling vision of a totalitarian future society.",
        "the great gatsby": "The Great Gatsby by F. Scott Fitzgerald - Literary Fiction. A portrait of the Jazz Age and the American Dream.",
    }

    title_key = title.lower().strip()

    if title_key in books:
        return books[title_key]
    else:
        return f"Book '{title}' not found in database. Try: Dune, Pride and Prejudice, The Hobbit, 1984, or The Great Gatsby."


print(f"Book tool ready: {lookup_book.name}")

Book tool ready: lookup_book


## Step 4: Create the Agent

Build the ReAct agent with all three tools and connect it to the memory store.

In [8]:
def create_book_agent(model, memory_tools: tuple, book_tool, store: InMemoryStore):
    """
    Create a ReAct agent with memory and book lookup capabilities.

    The agent should have access to:
    - The manage memory tool (for saving preferences)
    - The search memory tool (for retrieving preferences)
    - The book lookup tool (for finding book information)

    Args:
        model: The language model to use
        memory_tools: Tuple of (manage_tool, search_tool)
        book_tool: The book lookup tool
        store: The memory store for the agent to use

    Returns:
        A configured ReAct agent
    """
    # TODO: Combine all tools into a list

    # TODO: Create and return the agent using create_react_agent with the store
    tools = [*memory_tools,book_tool]
    agent = create_react_agent(
        model=model,
        tools=tools,
        store=store
    )
    return agent


# Create the agent (only if memory tools are implemented)
if manage_tool and search_tool:
    book_agent = create_book_agent(
        model=model,
        memory_tools=(manage_tool, search_tool),
        book_tool=lookup_book,
        store=memory_store
    )
    if book_agent:
        print("Agent created successfully!")
    else:
        print("Agent not implemented yet")
else:
    book_agent = None
    print("Complete the memory tools first")

Agent created successfully!


/tmp/ipykernel_16138/3344703975.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## Run Your Implementation

Test your agent with these sample interactions. The agent should save preferences and recall them later.

In [9]:
# Helper function to chat with the agent
def chat(message: str):
    """Send a message to the agent and display the response."""
    if not book_agent:
        print("Agent not ready. Complete the implementation first.")
        return

    print(f"\n{'='*60}")
    print(f"USER: {message}")
    print(f"{'='*60}")

    result = book_agent.invoke({
        "messages": [{"role": "user", "content": message}]
    })

    final_message = result["messages"][-1]
    print(f"\nAGENT: {final_message.content}\n")


# Test 1: Tell the agent your preferences
chat("Hi! I'm Alex. I love science fiction and fantasy books. My favorite author is Frank Herbert.")


USER: Hi! I'm Alex. I love science fiction and fantasy books. My favorite author is Frank Herbert.

AGENT: Hi Alex! It's great to meet a fellow fan of science fiction and fantasy. Frank Herbert is an amazing author. Do you have a favorite book by him?



In [10]:
# Test 2: Ask the agent to recall your preferences
chat("What genres do I like?")


USER: What genres do I like?

AGENT: You like science fiction and fantasy genres. Your favorite author is Frank Herbert.



In [11]:
# Test 3: Ask for a book recommendation based on preferences
chat("Can you look up a book that matches my taste?")


USER: Can you look up a book that matches my taste?

AGENT: Of course! To help me find a book that matches your taste, could you please tell me about your preferences? For example, your favorite genres, authors, or any specific themes or topics you enjoy.



## Test Memory Deletion

Test that the agent can delete memories when asked. The `manage_memory` tool supports delete operations.

**How deletion works:** The agent must:
1. First search for the memory to get its ID
2. Then call `manage_memory` with `action="delete"` and the memory `id`

In [12]:
# Test 4: Ask the agent to forget a preference
chat("Actually, I changed my mind. Please forget that Frank Herbert is my favorite author.")


USER: Actually, I changed my mind. Please forget that Frank Herbert is my favorite author.

AGENT: I've updated your preferences and removed Frank Herbert as your favorite author. If there's anything else you'd like to update or remember, just let me know!



In [13]:
# Test 5: Verify the memory was deleted
chat("Who is my favorite author?")


USER: Who is my favorite author?

AGENT: I don't have a record of your favorite author. If you'd like to share who it is, I can remember it for future reference.

